# Chapter 6. Density Functional Theory

**Research question:** if two calculations predict different molecular dipoles, did we change the physical approximation, the basis, or only the numerical integration? This matters when selecting a model for polarization, intermolecular interactions, or a reaction-energy study. We will change one ingredient at a time on the same small molecule.

Density functional theory (DFT) describes electronic structure using the electron density as a central variable. In practice, Kohn-Sham DFT still uses orbitals, and its accuracy depends on the chosen density functional approximation (DFA), basis, numerical settings, and physical model.

**Learning objectives**

After this chapter, you should be able to:

1. State what the Hohenberg-Kohn theorems establish, and what they do not provide.
2. Write a consistently normalized electron density and explain the Kohn-Sham energy terms.
3. Classify common functional families and distinguish local Kohn-Sham from generalized Kohn-Sham equations.
4. Recognize functional, basis, grid, self-consistency, and geometry errors as separate issues.
5. Run checked DFT single points and an actual geometry optimization with Psi4.
6. Compare methods on the same system without treating a lower approximate total energy as an accuracy ranking.

**Prerequisites:** Chapters 4 and 5: electronic wavefunctions, atomic units, basis sets, spin, and self-consistent-field calculations. Examples here concern isolated, nonrelativistic molecules within the clamped-nuclei Born-Oppenheimer approximation. They omit solvent and thermal effects.

**First reading:** follow density normalization, the Kohn-Sham energy inventory, and the water calculations. Read the theorem qualifications and functional taxonomy more slowly on a second pass. You do not need to derive a functional to use one carefully.

**Vocabulary:** a *function* such as $n(\mathbf r)$ takes a position and returns a density. A *functional* such as $E[n]$ takes a whole density profile and returns one number, an energy. A *grid* is a set of numerical integration points; a *basis* is a set of functions used to express orbitals. They are different approximations.


## 6.1. From a wavefunction to a density

Think of a density map as an electron-population map: integrating over a region gives the expected number of electrons in that region. It does not assign identifiable electrons to locations.

The interacting wavefunction depends on the coordinates and spins of all $N$ electrons. The spin-summed electron density $n(\mathbf r)$ depends on just three spatial coordinates:

$$n(\mathbf r)\geq0,\qquad \int n(\mathbf r)\,d\mathbf r=N.$$

Density is **electron number per volume**, not a normalized one-electron probability density unless $N=1$. In atomic units its usual unit is $a_0^{-3}$, where $a_0$ is the bohr. The electronic charge density is $-e n(\mathbf r)$.

For a nondegenerate ground state, the density contains sufficient information in principle to determine ground-state properties. This does not mean that formulas for every property are known, nor that an approximate density is exact. Ground-state DFT alone does not supply an excitation spectrum or reaction dynamics. See [Kohn's Nobel lecture](https://www.nobelprize.org/uploads/2018/06/kohn-lecture.pdf).

### A normalization check: hydrogen 1s

For a hydrogen atom with $Z=1$, the exact ground-state density in atomic units is $n(r)=e^{-2r}/\pi$. Because this density is spherically symmetric, the number of electrons in a shell of thickness $dr$ is $4\pi r^2n(r)\,dr$. The density and radial distribution are different quantities.

In [ ]:
import os
os.environ.setdefault("MKL_THREADING_LAYER", "SEQUENTIAL")
from IPython import get_ipython
if get_ipython() is not None:
    get_ipython().run_line_magic("matplotlib", "inline")
else:
    import matplotlib
    matplotlib.use("Agg")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

radius_bohr = np.linspace(0.0, 15.0, 10001)
hydrogen_density = np.exp(-2 * radius_bohr) / np.pi
radial_distribution = 4 * np.pi * radius_bohr**2 * hydrogen_density
integrated_electrons = np.trapezoid(radial_distribution, radius_bohr)
assert np.isclose(integrated_electrons, 1.0, atol=1e-7)
print(f"Integrated electron count: {integrated_electrons:.8f}")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.4), constrained_layout=True)
axes[0].plot(radius_bohr, hydrogen_density)
axes[0].set(xlim=(0, 6), xlabel="Radius (bohr)", ylabel="Density (electrons / bohr cubed)",
            title="Hydrogen 1s density")
axes[1].plot(radius_bohr, radial_distribution)
axes[1].set(xlim=(0, 6), xlabel="Radius (bohr)", ylabel="Radial distribution (1 / bohr)",
            title="Spherical-shell volume changes the shape")
for axis in axes:
    axis.grid(alpha=0.25)
plt.show()

### 6.1.1. Why density can be enough: the Hohenberg-Kohn statements

The remarkable claim is about information: for the ground state, a three-dimensional density can encode what appears to require a many-electron wavefunction. It is not a recipe for computing the answer. Here $v_{\mathrm{ext}}$ is the potential energy experienced by one electron due to the nuclei; $F$ collects the internal electronic energy.

Consider a fixed electron number, a fixed electron-electron interaction, and a scalar external potential $v_{\mathrm{ext}}(\mathbf r)$.

1. **Density determines the potential.** In the usual nondegenerate ground-state formulation, the ground-state density determines the external potential up to an additive constant. It therefore determines the electronic Hamiltonian and ground-state wavefunction up to its irrelevant overall phase. Degenerate ground states require a more careful formulation; a density need not identify a unique member of a degenerate set.
2. **A density variational principle exists.** With the exact universal internal-energy functional $F[n]$, minimizing

$$E_v[n]=F[n]+\int v_{\mathrm{ext}}(\mathbf r)n(\mathbf r)\,d\mathbf r$$

over the appropriate admissible densities gives the electronic ground-state energy. The universal part is $F[n]$, **not** the entire energy, which also depends on $v_{\mathrm{ext}}$.

These are existence and variational statements, not an explicit solution for $F[n]$. See [Hohenberg and Kohn, 1964](https://doi.org/10.1103/PhysRev.136.B864).

### 6.1.2. Deeper reading: which trial densities are admissible?

Normalization and nonnegativity are necessary, but one cannot simply insert any arbitrary function into a rigorous variational search. One useful formulation is Levy's constrained search:

$$F[n]=\min_{\Psi\rightarrow n}\langle\Psi|\hat T+\hat W_{ee}|\Psi\rangle.$$

The minimization is over normalized antisymmetric $N$-electron wavefunctions yielding $n$, with finite required energy expectations. Such a density is called **N-representable**. This construction avoids requiring every trial density to be the ground-state density of some external potential; ensemble extensions handle additional generality. See [Levy, 1979](https://pmc.ncbi.nlm.nih.gov/articles/PMC411802/).

The exact variational bound applies to the **exact functional** on its allowed domain. An approximate DFT total energy is not generally an upper bound to the exact energy. Comparing PBE and PBE0 total energies and choosing the more negative one is therefore not a valid way to select the more accurate functional.

For molecular calculations, add the nucleus-nucleus repulsion $E_{NN}$ to the electronic energy. It is constant at fixed nuclear geometry but changes during geometry optimization.

## 6.2. Kohn-Sham theory

### 6.2.1. Separating the energy

**Intuition first:** replace the difficult interacting-electron problem with an auxiliary orbital problem that has the same density. This makes most of the kinetic energy easy to calculate; the remaining exchange and correlation physics goes into a functional we must approximate.

Read the energy equation as an inventory:

| Term | Physical role |
|---|---|
| $T_s$ | Kinetic energy of the auxiliary noninteracting electrons |
| $\int v_{\mathrm{ext}}n$ | Electron attraction to the fixed nuclei |
| $E_H$ | Classical electrostatic repulsion of the density with itself |
| $E_{xc}$ | Correction for exchange, correlation, and the missing kinetic contribution |
| $E_{NN}$ | Repulsion between the nuclei |

Kohn and Sham introduced an auxiliary noninteracting system that reproduces the interacting density, where an appropriate noninteracting representation exists. Its orbitals make the dominant kinetic-energy contribution tractable:

$$E[n]=T_s[n]+\int v_{\mathrm{ext}}(\mathbf r)n(\mathbf r)\,d\mathbf r
+E_H[n]+E_{xc}[n]+E_{NN},$$

$$E_H[n]=\frac12\iint\frac{n(\mathbf r)n(\mathbf r')}{|\mathbf r-\mathbf r'|}\,d\mathbf r\,d\mathbf r'.$$

Here $T_s$ is the auxiliary noninteracting kinetic energy and $E_H$ is the classical Coulomb energy of the density. By definition,

$$E_{xc}[n]=\big(T[n]-T_s[n]\big)+\big(W_{ee}[n]-E_H[n]\big).$$

Thus exchange-correlation includes the **kinetic correlation** $T-T_s$, as well as the difference between the true electron-electron interaction and classical Hartree energy. It is not merely a small additional electrostatic term. Exact DFT would include correlation exactly; practical DFAs approximate it. See [Kohn and Sham, 1965](https://doi.org/10.1103/PhysRev.140.A1133).

### 6.2.2. Equations and occupations

The structure resembles the HF orbital equation in Chapter 5: apply an effective one-electron operator to an orbital $\phi_i$ and obtain its eigenvalue $\epsilon_i$. Here $i$ labels the orbital, $\nabla^2$ measures spatial curvature, $\sigma$ labels spin, and $f$ is an occupation number. The functional derivative $\delta E_{xc}/\delta n$ describes how the energy responds to a small local change in the density. You can follow the calculations without deriving this derivative.

All equations in this section use **atomic units**, so $\hbar=m_e=e=4\pi\epsilon_0=1$ in the corresponding atomic-unit convention. For a local multiplicative Kohn-Sham potential,

$$\left[-\frac12\nabla^2+v_{\mathrm{ext}}(\mathbf r)+v_H(\mathbf r)+v_{xc}(\mathbf r)\right]
\phi_i(\mathbf r)=\epsilon_i\phi_i(\mathbf r),$$

$$v_H(\mathbf r)=\int\frac{n(\mathbf r')}{|\mathbf r-\mathbf r'|}\,d\mathbf r',\qquad
v_{xc}(\mathbf r)=\frac{\delta E_{xc}[n]}{\delta n(\mathbf r)}.$$

For clamped nuclei, $v_{\mathrm{ext}}(\mathbf r)=-\sum_A Z_A/|\mathbf r-\mathbf R_A|$. The density includes occupation numbers:

$$n(\mathbf r)=\sum_{\sigma\in\{\alpha,\beta\}}\sum_i f_{i\sigma}|\phi_{i\sigma}(\mathbf r)|^2.$$

For ordinary integer occupations, $f_{i\sigma}$ is 0 or 1. A closed-shell singlet with doubly occupied spatial orbitals instead has $n=2\sum_{i=1}^{N/2}|\phi_i|^2$. Omitting this factor of two counts only half the electrons. Fractional occupations and spin-dependent functionals are useful in other settings, but are not needed for our closed-shell molecular examples.

The Kohn-Sham determinant is an auxiliary construction; it is not the exact correlated many-electron wavefunction. Orbital eigenvalues are not generally excitation energies. In particular, a HOMO-LUMO eigenvalue gap should not be labeled an optical absorption energy or an experimental fundamental gap.

### 6.2.3. Self-consistency

The potential depends on the density, and the orbitals determine the density. A practical loop is:

1. Choose nuclear coordinates, charge, spin, basis, functional, and numerical settings.
2. Guess a density matrix.
3. Build the Coulomb and exchange-correlation contributions.
4. Solve the orbital equations and apply the chosen occupations.
5. Update or mix the density; check energy and density/orbital residual convergence.
6. Repeat until the stated criteria are met.

Converged SCF means the electronic iteration met its numerical criteria. It does **not** establish that the lowest electronic solution, correct spin state, or optimized nuclear geometry has been found. Geometry optimization is an outer loop that changes nuclear coordinates using gradients and runs a new SCF calculation at each step. See [Psi4's SCF documentation](https://psi4.github.io/psi4docs/master/scf.html).

## 6.3. Exchange-correlation approximations

### 6.3.1. A useful classification: Jacob's ladder (reference table)

The ladder organizes approximations by the ingredients they use. It is not a guarantee that each higher rung improves every result. Cost also depends on the implementation, basis, and system.

| Rung | Additional information used | Examples |
|---|---|---|
| 1: LDA / LSDA | Density at each point, or separate spin densities | Uniform-electron-gas based approximations |
| 2: GGA | Density gradients | PBE, BLYP |
| 3: meta-GGA | Usually orbital kinetic-energy density $\tau$, sometimes density Laplacians | SCAN, r2SCAN |
| 4: hybrid | Occupied-orbital nonlocal exact-exchange contributions | PBE0, B3LYP; many range-separated hybrids |
| 5: unoccupied-orbital dependent | For example, perturbative or response-based correlation | Double hybrids; random-phase approximation approaches |

LDA is local; GGAs and meta-GGAs are often called **semilocal**, although the usual $\tau$ ingredient is orbital dependent. A range-separated hybrid splits the electron-electron interaction by distance; range separation is not a separate rung above all other hybrids. A dispersion correction is also not a ladder rung. See [Perdew and Schmidt's ladder](https://doi.org/10.1063/1.1390175) and the [original nonempirical meta-GGA construction](https://arxiv.org/abs/cond-mat/0306203).

### 6.3.2. Hybrids and generalized Kohn-Sham equations

A simple global hybrid mixes a fraction $a$ of exact exchange evaluated with the auxiliary orbitals:

$$E_{xc}^{\mathrm{hybrid}}=aE_x^{\mathrm{HF}}+(1-a)E_x^{\mathrm{DFA}}+E_c^{\mathrm{DFA}}.$$

For PBE0, $a=0.25$ and the remaining exchange and correlation are from PBE. This is a specific construction, not a formula for every hybrid; B3LYP, for example, uses a different mixture. See [Adamo and Barone's PBE0 model](https://doi.org/10.1063/1.478522).

Orbital variation of a hybrid normally introduces a **nonlocal exchange operator**, leading to generalized Kohn-Sham equations. The simple local $v_{xc}(\mathbf r)$ equation above is therefore not the whole operator used in a routine hybrid calculation. Orbital-dependent meta-GGAs are also commonly implemented in a generalized Kohn-Sham framework. Using an optimized effective potential is a distinct way to retain a local potential.

Exact exchange is the determinant exchange contribution. It does not make a hybrid's total energy or electron correlation exact. Hybrids often cost more than semilocal DFAs and can reduce some errors without removing all of them. The functional name, range-separation parameters, and any dispersion variant belong in the reported method. See [Psi4's DFT implementation and functional selection](https://psi4.github.io/psi4docs/master/dft.html#functional-selection).

## 6.4. DFT, wavefunction methods, and sources of error

### 6.4.1. DFT is not simply the opposite of ab initio

Hartree-Fock and post-Hartree-Fock are **wavefunction methods**; Kohn-Sham DFT uses a density-functional formulation with auxiliary orbitals. Both are quantum electronic-structure approaches and both involve practical approximations. The term *ab initio* describes a first-principles approach, not an exclusive synonym for wavefunction methods.

Hartree-Fock contains determinant exchange but omits electron correlation in its usual ground-state energy. Practical DFT approximates exchange and correlation together; some functionals are constructed largely from exact constraints, while others contain fitted parameters. Semilocal DFT is often less costly than highly correlated wavefunction methods for a similar system, but no universal accuracy or speed ordering covers all methods and molecules.

Increasing the basis and tightening numerical settings improves a calculation of a **chosen approximation**; it does not systematically remove that approximation's functional error. A useful benchmark compares a defined observable, not just the magnitude of total electronic energies.

### 6.4.2. Important failure modes

| Issue | What can go wrong | What to investigate |
|---|---|---|
| Functional error | Approximate energies or densities can be inaccurate even after numerical convergence | Benchmarks for the target property and chemical domain |
| One-electron self-interaction | Hartree self-repulsion is incompletely canceled; exact one-electron $E_{xc}=-E_H$ | Suitable exact constraints and validated functionals |
| Delocalization error | Fractional charge can be spuriously favored; charge distributions and barriers can be distorted | Charge localization tests and appropriate hybrid/range-separated models |
| Static correlation | Several configurations become important, for example in stretched bonds | State character, spin, and multireference or other suitable methods |
| Missing long-range dispersion | Semilocal DFAs do not recover the asymptotic interaction of separated fragments | A compatible D3/D4 correction or nonlocal correlation treatment |
| Numerical/model error | A small basis, coarse grid, unconverged SCF, wrong geometry, or wrong charge/spin changes results | Independent checks of each setting and input assumption |

Self-interaction and delocalization are related concepts, but they are not interchangeable labels for every functional error. Common hybrids do not solve static correlation automatically. Exact ground-state DFT includes dispersion; the failure belongs to particular approximations. An empirical dispersion add-on does not correct every other error, and its damping/parameterization must match the functional.

Primary discussions: [delocalization error](https://pubmed.ncbi.nlm.nih.gov/18518055/), [fractional spins and static correlation](https://pubmed.ncbi.nlm.nih.gov/19044996/), and the [D3 authors' documentation](https://www.chemie.uni-bonn.de/grimme/de/software/dft-d3).

## 6.5. Software and its role

Capabilities and licensing are different questions. A molecular viewer or graphical interface also need not be the engine doing an electronic-structure calculation.

| Software | Typical role | Access/licensing context |
|---|---|---|
| [Psi4](https://github.com/psi4/psi4), [PySCF](https://pyscf.org/about.html), [NWChem](https://nwchemgit.github.io/) | Molecular electronic structure, including DFT; some also support periodic systems | Open source, under each project's license |
| [Gaussian](https://gaussian.com/), [Q-Chem](https://www.q-chem.com/purchase/license/) | General molecular electronic structure, including DFT | Commercially licensed software |
| [ORCA](https://www.faccts.de/orca/) | Molecular electronic structure and spectroscopy | Academic access is available under its terms; commercial use has separate licensing |
| [Quantum ESPRESSO](https://www.quantum-espresso.org/) | Primarily periodic plane-wave/pseudopotential calculations | Open-source suite |
| [VASP](https://www.vasp.at/) | Primarily periodic electronic structure | Licensed software |

The exact method, basis/pseudopotentials, grids, and convergence settings still need to be specified within any package. MOPAC is primarily a **semiempirical quantum-chemistry** program, introduced in [Chapter 7](Chapter07.ipynb); it should not be presented as an interchangeable general-purpose Kohn-Sham DFT engine. See the [MOPAC project](https://openmopac.net/).

## 6.6. Reproducible molecular DFT with Psi4

We will perform three connected tasks:

1. Compare PBE and PBE0 on the **same fixed water geometry**, visualize their densities, and separate functional changes from grid and basis changes.
2. Optimize that water geometry with PBE and verify gradients and vibrational curvature.
3. Calculate one B3LYP single-point energy for a locally prepared 1,3-butadiene conformer.

The first two tasks are deliberately small so numerical checks remain practical on a laptop. The butadiene example illustrates the conversion from an RDKit molecular graph to a quantum-chemistry input. A single-point energy evaluates a specified geometry. Geometry optimization changes nuclear coordinates using energy gradients.

Use the **quantum-chemistry environment in the repository README**, containing Psi4, NumPy, pandas, Matplotlib, and RDKit, and choose its Jupyter kernel. Package installation belongs in a terminal, not an executable notebook cell. All molecular inputs and basis data used here are local once the environment is installed. See [Psi4 installation and Python usage](https://psicode.org/psi4manual/master/psiapi.html).

### 6.6.1. Set and record the calculation model

The baseline uses **def2-SVP**, a small polarized double-zeta orbital basis. This is a classroom basis, not a claim of basis convergence. We choose RKS for neutral, closed-shell singlets. The integration grid has 50 radial shells and up to 194 angular points before the specified robust pruning. A numerical grid and an orbital basis serve different purposes: quadrature evaluates exchange-correlation integrals; basis functions represent orbitals.

`SCF_TYPE='DF'` means **density fitting** of electron-repulsion integrals, not density functional theory. The `def2-universal-jkfit` auxiliary basis is separate from the orbital basis. The SCF energy and density convergence settings are explicit, and nonconvergence raises an error. See [Psi4 grid controls](https://psi4.github.io/psi4docs/master/dft.html#grid-selection) and [density-fitting options](https://psi4.github.io/psi4docs/master/scf.html#density-fitted-integrals).

Logs and generated geometries go into `outputs/chapter06/`; running this notebook again replaces its named output files. Psi4 scratch files use a chapter-specific subdirectory. One CPU thread and 512 MiB keep this example modest; an exact trajectory of floating-point results can still depend on software/build versions.

For this small teaching system we use a bounded 50-radial / 194-angular-point grid with robust pruning. We explicitly test a finer grid before interpreting a dipole difference. This choice keeps the repeated gradients short; it is not a default recommendation for other molecules or sensitive properties.

In [ ]:
from pathlib import Path
from importlib.metadata import version
import json
import psi4

OUTPUT_DIR = Path("outputs/chapter06").resolve()
SCRATCH_DIR = OUTPUT_DIR / "scratch"
SCRATCH_DIR.mkdir(parents=True, exist_ok=True)
psi4.core.clean_options()
psi4.core.clean_variables()
psi4.core.IOManager.shared_object().set_default_path(str(SCRATCH_DIR))
psi4.core.set_output_file(str(OUTPUT_DIR / "psi4.log"), False)
psi4.set_num_threads(1)
psi4.set_memory("512 MiB")

BASE_OPTIONS = {
    "basis": "def2-svp",
    "reference": "rks",
    "scf_type": "df",
    "df_basis_scf": "def2-universal-jkfit",
    "e_convergence": 1e-9,
    "d_convergence": 1e-8,
    "maxiter": 100,
    "fail_on_maxiter": True,
    "dft_radial_points": 50,
    "dft_spherical_points": 194,
    "dft_pruning_scheme": "robust",
    "g_convergence": "gau_tight",
    "geom_maxiter": 50,
}
psi4.set_options(BASE_OPTIONS)
versions = {package: version(package) for package in ("numpy", "pandas", "matplotlib", "rdkit")}
versions["psi4"] = psi4.__version__
print("Versions:", versions)
print("Orbital / auxiliary basis:", BASE_OPTIONS["basis"], "/", BASE_OPTIONS["df_basis_scf"])
print("Output folder:", OUTPUT_DIR.name)

### 6.6.2. Keep the geometry fixed when comparing functionals

Our initial water geometry is deliberately approximate: both O-H distances are 1.00 angstrom and the H-O-H angle is 110 degrees. It is an illustrative input, not an experimental geometry. Charge 0 and multiplicity 1 specify a neutral singlet with ten electrons.

The Z-matrix below specifies distances and an angle directly; it is not an XYZ file with an atom-count header. `symmetry c1` avoids symmetry-block bookkeeping in the teaching analysis. We clone the same molecule for each single point and keep an unchanged reference for the later optimization comparison.

In [ ]:
water_start = psi4.geometry("""
0 1
O
H 1 1.00
H 1 1.00 2 110.0
units angstrom
symmetry c1
no_reorient
no_com
""")
water_start.update_geometry()
water_reference_bohr = np.array(water_start.geometry().np, copy=True)
assert water_start.natom() == 3
assert water_start.molecular_charge() == 0 and water_start.multiplicity() == 1
print("Water atoms / charge / multiplicity:", water_start.natom(), 0, 1)

**Inspect the returned wavefunction object.** Psi4 uses this object to store orbitals and density matrices even for DFT; its name does not imply an exact correlated wavefunction. In a nonorthogonal atomic-orbital basis with overlap matrix $S$, the electron count is

$$N=\mathrm{Tr}[(D_\alpha+D_\beta)S].$$

The ordinary trace of the density matrix alone is generally incorrect in that basis. Our helper checks the count and saves the total clamped-nuclei energy, dipole magnitude, and eigenvalue gap. `SCF DIPOLE` is in atomic dipole units, so we convert it to debye. These are different units from bohr and hartree. See [Psi4's property definitions](https://psi4.github.io/psi4docs/master/glossary_psivariables.html#psivar-SCF-DIPOLE).

In [ ]:
def checked_single_point(functional, molecule):
    """Run a closed-shell single point; SCF nonconvergence must raise an error."""
    energy_hartree, wfn = psi4.energy(functional, molecule=molecule, return_wfn=True)
    expected_electrons = sum(molecule.Z(i) for i in range(molecule.natom())) - molecule.molecular_charge()
    density_total = np.array(wfn.Da().np + wfn.Db().np, copy=True)
    overlap = np.array(wfn.S().np, copy=True)
    electron_count = np.trace(density_total @ overlap)
    assert np.isfinite(energy_hartree) and np.isfinite(density_total).all()
    assert np.isclose(electron_count, expected_electrons, atol=1e-7)
    assert wfn.nalpha() == wfn.nbeta(), "This analysis expects a closed-shell reference"
    orbital_energies = np.asarray(wfn.epsilon_a().np)
    homo_index = wfn.nalpha() - 1
    gap_ev = (orbital_energies[homo_index + 1] - orbital_energies[homo_index]) * psi4.constants.hartree2ev
    dipole_debye = np.linalg.norm(np.asarray(wfn.variable("SCF DIPOLE"))) * psi4.constants.dipmom_au2debye
    row = {"functional": functional.upper(), "energy_hartree": float(energy_hartree),
           "dipole_debye": float(dipole_debye), "orbital_gap_eV": float(gap_ev),
           "electrons": float(electron_count), "SCF_iterations": int(wfn.variable("SCF ITERATIONS"))}
    assert np.isfinite([row["dipole_debye"], row["orbital_gap_eV"]]).all()
    return row, wfn

water_rows, water_wavefunctions = [], {}
for functional in ("pbe", "pbe0"):
    row, wfn = checked_single_point(functional, water_start.clone())
    water_rows.append(row)
    water_wavefunctions[functional] = wfn
water_comparison = pd.DataFrame(water_rows).set_index("functional")
assert np.isclose(water_wavefunctions["pbe0"].functional().x_alpha(), 0.25)
np.testing.assert_allclose(water_start.geometry().np, water_reference_bohr, atol=1e-12)
display(water_comparison.round({"energy_hartree": 9, "dipole_debye": 5, "orbital_gap_eV": 4}))

The methods give different dipoles and orbital gaps because they produce different approximate electronic structures on the same geometry. Neither the lower total energy nor a larger orbital gap establishes greater accuracy. This table shows **method dependence**, not a benchmark against experiment. The gap is an orbital eigenvalue difference, not an absorption energy.

Total energies are most naturally retained in hartree. Meaningful energy differences, such as geometry relaxation at the **same level of theory**, can be converted to kJ/mol. The energies here include nucleus-nucleus repulsion and exclude zero-point and thermal corrections.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.4), constrained_layout=True)
labels = water_comparison.index.tolist()
axes[0].bar(labels, water_comparison.dipole_debye, color=["#28788e", "#b45a36"])
axes[0].set(ylabel="Dipole magnitude (debye)", title="Same fixed water geometry")
axes[1].bar(labels, water_comparison.orbital_gap_eV, color=["#28788e", "#b45a36"])
axes[1].set(ylabel="HOMO-LUMO eigenvalue gap (eV)", title="Not an optical excitation energy")
for axis in axes:
    axis.grid(axis="y", alpha=0.25)
    axis.set_axisbelow(True)
fig.savefig(OUTPUT_DIR / "fixed_geometry_properties.png", dpi=150)
plt.show()

### See what changed: a molecular-plane density slice

The bars summarize two properties. A spatial map asks a different question: **where did the electronic distribution change?** We evaluate both total spin-summed density matrices in their atomic-orbital basis:

$$n(\mathbf r)=\sum_{\mu\nu}D_{\mu\nu}\chi_\mu(\mathbf r)\chi_\nu(\mathbf r).$$

Here $D=D^\alpha+D^\beta$ includes both spins; $\chi_\mu$ is basis function $\mu$ evaluated at position $\mathbf r$. These real-valued molecular calculations need no complex conjugates. The first two panels use the same logarithmic color scale to reveal both dense and diffuse regions; the third uses a signed linear scale to show PBE0 minus PBE. A brighter region is not an identifiable electron or evidence of greater accuracy.

This is a **two-dimensional slice through the molecule**, with density in electrons per bohr cubed. The slice integral is not the electron count; the basis-space trace above already verifies ten electrons. The plane axes are defined from the O-H bonds, so no assumed laboratory orientation is required. See the [Psi4 basis-evaluation API](https://psicode.org/psi4manual/master/api/psi4.core.BasisSet.html#psi4.core.BasisSet.compute_phi) and [official AO evaluation example](https://github.com/psi4/psi4/blob/master/samples/phi-ao/input.dat).

In [ ]:
from matplotlib.colors import LogNorm, TwoSlopeNorm

origin = water_reference_bohr[0]
plane_x = water_reference_bohr[1] - origin
plane_x /= np.linalg.norm(plane_x)
plane_y = water_reference_bohr[2] - origin
plane_y -= np.dot(plane_y, plane_x) * plane_x
plane_y /= np.linalg.norm(plane_y)
atom_plane = (water_reference_bohr - origin) @ np.column_stack([plane_x, plane_y])
u = np.linspace(atom_plane[:, 0].min() - 2.5, atom_plane[:, 0].max() + 2.5, 75)
v = np.linspace(atom_plane[:, 1].min() - 2.5, atom_plane[:, 1].max() + 2.5, 75)
U, V = np.meshgrid(u, v)
points_bohr = origin + U.ravel()[:, None] * plane_x + V.ravel()[:, None] * plane_y
density_slices = {}
for functional, wfn in water_wavefunctions.items():
    basis = wfn.basisset()
    phi = np.asarray([basis.compute_phi(*point) for point in points_bohr])
    D = wfn.Da().np + wfn.Db().np
    values = np.einsum("pi,ij,pj->p", phi, D, phi, optimize=True)
    assert phi.shape == (75 * 75, basis.nbf())
    assert np.isfinite(values).all() and values.min() > -1e-10
    density_slices[functional] = values.reshape(U.shape)

difference = density_slices["pbe0"] - density_slices["pbe"]
limit = np.abs(difference).max()
assert limit > 0
fig, axes = plt.subplots(1, 3, figsize=(12, 3.8), layout="constrained")
common_norm = LogNorm(vmin=1e-3, vmax=max(a.max() for a in density_slices.values()))
for axis, functional in zip(axes[:2], ("pbe", "pbe0")):
    mesh = axis.pcolormesh(U, V, density_slices[functional], norm=common_norm,
                           cmap="viridis", shading="auto")
    axis.set_title(functional.upper() + " density")
fig.colorbar(mesh, ax=axes[:2], label="Electrons / bohr cubed", shrink=0.8)
change = axes[2].pcolormesh(U, V, difference, norm=TwoSlopeNorm(vmin=-limit, vcenter=0, vmax=limit),
                           cmap="RdBu_r", shading="auto")
axes[2].set_title("PBE0 minus PBE")
fig.colorbar(change, ax=axes[2], label="Density change", shrink=0.8)
for axis in axes:
    axis.scatter(*atom_plane.T, s=24, c="white", edgecolors="black", zorder=3)
    for symbol, (a, b) in zip(("O", "H", "H"), atom_plane):
        axis.annotate(symbol, (a, b), xytext=(5, 5), textcoords="offset points",
                      color="black", bbox={"facecolor": "white", "alpha": 0.8, "pad": 1})
    axis.set(xlabel="In-plane coordinate u (bohr)", ylabel="v (bohr)", aspect="equal")
fig.savefig(OUTPUT_DIR / "water_density_slices.png", dpi=150)
plt.show()

### 6.6.3. Change the integration grid without changing the functional

Now repeat **PBE0 at the same geometry and basis** with 150 radial shells and up to 974 angular points. This isolates grid sensitivity; the PBE-to-PBE0 difference above did not. The assertion below applies a classroom tolerance to this one example. It is not a universal grid recommendation or a proof that every property is converged. Gradients and meta-GGAs can require additional care.

In [ ]:
try:
    psi4.set_options({"dft_radial_points": 150, "dft_spherical_points": 974})
    fine_grid_row, _ = checked_single_point("pbe0", water_start.clone())
finally:
    # Restore the exact baseline settings before subsequent calculations.
    psi4.set_options(BASE_OPTIONS)

grid_delta_hartree = fine_grid_row["energy_hartree"] - water_comparison.loc["PBE0", "energy_hartree"]
grid_delta_kj_mol = grid_delta_hartree * psi4.constants.hartree2kJmol
assert abs(grid_delta_hartree) < 1e-5, "Grid sensitivity exceeds the example's 1e-5 hartree tolerance"
print(f"PBE0 grid change, fine minus baseline: {grid_delta_hartree:.3e} hartree")
print(f"Same change in molar energy units: {grid_delta_kj_mol:.6f} kJ/mol")

### A research decision: is a property numerically stable enough to compare?

Suppose we want to discuss dipole differences at a resolution of **0.001 D**. This is an illustrative reporting target, not an experimental uncertainty. A grid change tests integration sensitivity at a fixed physical approximation. A larger orbital basis tests another numerical representation; changing PBE to PBE0 changes the exchange-correlation approximation.

The next calculation enlarges only the PBE0 orbital basis from def2-SVP to def2-TZVP at the original geometry. The same universal fitting basis and baseline integration grid are retained. Each row below is a controlled comparison with the same reference, PBE0/def2-SVP on the baseline grid. These changes need not be additive; none is a certified error bar. In particular, the more negative total energy from a different functional is not an accuracy score.

In [ ]:
try:
    psi4.set_options({"basis": "def2-tzvp"})
    larger_basis_row, _ = checked_single_point("pbe0", water_start.clone())
finally:
    psi4.set_options(BASE_OPTIONS)

reference_dipole = water_comparison.loc["PBE0", "dipole_debye"]
property_sensitivity = pd.DataFrame([
    {"change": "Grid only: 150 / 974", "dipole_D": fine_grid_row["dipole_debye"]},
    {"change": "Basis only: def2-TZVP", "dipole_D": larger_basis_row["dipole_debye"]},
    {"change": "Functional only: PBE", "dipole_D": water_comparison.loc["PBE", "dipole_debye"]},
]).set_index("change")
property_sensitivity["delta_from_reference_D"] = property_sensitivity.dipole_D - reference_dipole
property_sensitivity["absolute_change_D"] = property_sensitivity.delta_from_reference_D.abs()
assert np.isfinite(property_sensitivity.to_numpy()).all()
display(property_sensitivity.round(7))
grid_stable_at_target = property_sensitivity.loc["Grid only: 150 / 974", "absolute_change_D"] < 0.001
print(f"Grid change below the illustrative 0.001 D resolution: {grid_stable_at_target}")
print("This tests numerical stability, not agreement with experiment.")

fig, axis = plt.subplots(figsize=(8, 3.2), layout="constrained")
axis.barh(property_sensitivity.index, property_sensitivity.absolute_change_D,
          color=["#28788e", "#d68b35", "#9366a1"])
axis.axvline(0.001, color="black", linestyle="--", label="Illustrative 0.001 D resolution")
axis.set(xlabel="Absolute dipole change from PBE0 / def2-SVP (D)",
         title="One input changed at a time, same water geometry")
axis.legend(loc="lower right")
fig.savefig(OUTPUT_DIR / "dipole_sensitivity.png", dpi=150)
plt.show()

### 6.6.4. Actually optimize the water geometry

`psi4.optimize` moves the nuclei using energy gradients, unlike `psi4.energy`. We use PBE/def2-SVP with the same baseline grid and a tight geometry-convergence preset. Optimization failure should stop the notebook; reaching the maximum iteration count is not success. We then calculate a fresh gradient as an independent diagnostic. See [Psi4 geometry optimization](https://psi4.github.io/psi4docs/master/optking.html).

The energy lowering relative to the **initial PBE geometry** is a relaxation energy at this computational level. It is not a reaction energy, binding energy, enthalpy, or free energy.

In [ ]:
water_optimized = water_start.clone()
optimized_energy, optimized_wfn = psi4.optimize("pbe", molecule=water_optimized, return_wfn=True)
optimized_gradient = np.asarray(psi4.gradient("pbe", molecule=water_optimized).np)
maximum_gradient = float(np.max(np.abs(optimized_gradient)))
assert np.isfinite(optimized_energy) and np.isfinite(optimized_gradient).all()
assert maximum_gradient < 3e-5, "The final Cartesian gradient is larger than the teaching tolerance"
initial_pbe_energy = water_comparison.loc["PBE", "energy_hartree"]
assert optimized_energy < initial_pbe_energy
relaxation_kj_mol = (optimized_energy - initial_pbe_energy) * psi4.constants.hartree2kJmol
print(f"Optimized PBE/def2-SVP energy: {optimized_energy:.9f} hartree")
print(f"Relaxation energy (optimized minus initial): {relaxation_kj_mol:.4f} kJ/mol")
print(f"Largest Cartesian gradient component: {maximum_gradient:.3e} hartree/bohr")


def water_geometry_summary(coords_bohr):
    """Distances and angle for the O, H, H atom ordering used above."""
    coords_angstrom = np.asarray(coords_bohr) * psi4.constants.bohr2angstroms
    first, second = coords_angstrom[1] - coords_angstrom[0], coords_angstrom[2] - coords_angstrom[0]
    cosine = np.dot(first, second) / (np.linalg.norm(first) * np.linalg.norm(second))
    return {"O-H1_angstrom": np.linalg.norm(first), "O-H2_angstrom": np.linalg.norm(second),
            "H-O-H_degrees": np.degrees(np.arccos(np.clip(cosine, -1, 1)))}

geometry_comparison = pd.DataFrame([
    water_geometry_summary(water_reference_bohr),
    water_geometry_summary(water_optimized.geometry().np),
], index=["Initial input", "PBE optimized"])
display(geometry_comparison.round(5))
np.testing.assert_allclose(water_start.geometry().np, water_reference_bohr, atol=1e-12)

### 6.6.5. Check that the stationary point is a minimum

A small gradient can occur at a minimum or a saddle point. We compute the harmonic vibrational curvature at the optimized geometry. Water is nonlinear, so it has $3N-6=3$ vibrational modes. Translational/rotational modes can carry tiny numerical imaginary components and should not be counted as internal instabilities.

The calculation below requests a Hessian from **finite differences of analytic gradients** (`dertype=1`) and selects only the vibrational modes identified by Psi4. Positive vibrational curvature supports a **local minimum at the chosen computational level**. It does not prove a global minimum for a general molecule. Harmonic frequencies are model predictions, not exact experimental fundamentals; anharmonicity, basis, and functional errors remain. See [Psi4 vibrational analysis](https://psi4.github.io/psi4docs/master/freq.html).

In [ ]:
frequency_energy, frequency_wfn = psi4.frequency(
    "pbe", molecule=water_optimized, dertype=1, return_wfn=True,
)
frequencies = np.asarray(frequency_wfn.frequency_analysis["omega"].data)
mode_types = np.asarray(frequency_wfn.frequency_analysis["TRV"].data)
vibrational_frequencies = frequencies[mode_types == "V"]
assert len(vibrational_frequencies) == 3
assert np.isfinite(vibrational_frequencies).all()
assert np.max(np.abs(vibrational_frequencies.imag)) < 1e-3
assert np.all(vibrational_frequencies.real > 0), "An internal mode has nonpositive curvature"
assert abs(frequency_energy - optimized_energy) < 1e-6
frequency_table = pd.DataFrame({"mode": np.arange(1, 4),
                                "harmonic_wavenumber_cm-1": vibrational_frequencies.real})
display(frequency_table.round(2))
print("Three positive vibrational modes support a local minimum at PBE/def2-SVP.")

### What makes the optimized structure usable?

There are three separate checks: lower energy shows that relaxation occurred, a small gradient indicates stationarity, and positive internal vibrational curvatures support a local minimum. The geometry drawing below uses only the calculated bond lengths and angle, placed in a common plane for comparison. The harmonic frequencies are calculated properties of this small gas-phase model, not measured spectral lines; no intensities or anharmonic corrections are shown.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), layout="constrained")
for label, row in geometry_comparison.iterrows():
    theta = np.radians(row["H-O-H_degrees"])
    xy = np.array([[0., 0.], [row["O-H1_angstrom"], 0.],
                   [row["O-H2_angstrom"] * np.cos(theta), row["O-H2_angstrom"] * np.sin(theta)]])
    axes[0].plot(xy[[1, 0, 2], 0], xy[[1, 0, 2], 1], "o-", label=label)
axes[0].set(xlabel="x (angstrom)", ylabel="y (angstrom)", aspect="equal",
            title="Internal geometry, common orientation")
axes[0].legend()
axes[1].vlines(vibrational_frequencies.real, 0, 1, color="#28788e", linewidth=3)
axes[1].scatter(vibrational_frequencies.real, np.ones(3), color="#28788e")
axes[1].set(xlabel="Harmonic wavenumber (cm$^{-1}$)", yticks=[], ylim=(0, 1.2),
            title="Three positive internal modes")
fig.savefig(OUTPUT_DIR / "water_minimum_checks.png", dpi=150)
plt.show()

### 6.6.6. From a molecular graph to a butadiene single point

Return to the organic-molecule workflow: make **1,3-butadiene, C4H6**, add hydrogens, generate a seeded ETKDGv3 conformer, and relax it with UFF. UFF is a molecular-mechanics preparation step; it does **not** optimize the geometry at the later DFT level.

Build the Psi4 geometry explicitly from element-coordinate rows and specify **charge, multiplicity, and angstrom units**. A basic XYZ record's atom-count/comment header is not a charge/multiplicity specification. The central C-C torsion is printed to document which prepared conformer was used. This is one conformer, not a conformational search. See [RDKit 3D preparation](https://www.rdkit.org/docs/GettingStartedInPython.html#working-with-3d-molecules).

In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem, Draw, rdMolDescriptors, rdMolTransforms

butadiene_rdkit = Chem.AddHs(Chem.MolFromSmiles("C=CC=C"))
assert rdMolDescriptors.CalcMolFormula(butadiene_rdkit) == "C4H6"
embedding = AllChem.ETKDGv3()
embedding.randomSeed = 2026
embedding.numThreads = 1
assert AllChem.EmbedMolecule(butadiene_rdkit, embedding) == 0
assert AllChem.UFFHasAllMoleculeParams(butadiene_rdkit)
assert AllChem.UFFOptimizeMolecule(butadiene_rdkit, maxIters=1000) == 0
butadiene_coordinates = np.array(butadiene_rdkit.GetConformer().GetPositions(), copy=True)
central_torsion_deg = rdMolTransforms.GetDihedralDeg(butadiene_rdkit.GetConformer(), 0, 1, 2, 3)
print(f"Prepared C1-C2-C3-C4 torsion: {central_torsion_deg:.3f} degrees")

geometry_lines = ["0 1"]
for atom, (x, y, z) in zip(butadiene_rdkit.GetAtoms(), butadiene_coordinates):
    geometry_lines.append(f"{atom.GetSymbol()} {x:.10f} {y:.10f} {z:.10f}")
geometry_lines += ["units angstrom", "symmetry c1", "no_reorient", "no_com"]
butadiene_input = "\n".join(geometry_lines)
butadiene_qm = psi4.geometry(butadiene_input)
butadiene_qm.update_geometry()
assert butadiene_qm.natom() == 10
np.testing.assert_allclose(butadiene_qm.geometry().np * psi4.constants.bohr2angstroms,
                          butadiene_coordinates, atol=1e-8)
# This is a separate 2D graph depiction; it does not display the quantum input geometry.
butadiene_2d = Chem.RemoveHs(Chem.Mol(butadiene_rdkit))
AllChem.Compute2DCoords(butadiene_2d)
display(Draw.MolToImage(butadiene_2d, size=(420, 180)))

In [ ]:
psi4.set_options(BASE_OPTIONS)
butadiene_row, butadiene_wfn = checked_single_point("b3lyp", butadiene_qm)
assert np.isclose(butadiene_row["electrons"], 30.0)
display(pd.DataFrame([butadiene_row]).set_index("functional").round(7))
print("B3LYP/def2-SVP single point on a UFF-prepared butadiene geometry; no DFT optimization here.")

The absolute butadiene energy cannot be compared with the water energy to decide which molecule is more stable: the compositions differ. A chemical reaction or conformer comparison needs a well-defined, balanced comparison at consistent computational settings. No dispersion correction is applied to these isolated-molecule demonstrations; this is not a recommended protocol for dispersion-bound complexes.

The results distinguish three operations: preparing a geometry with UFF, evaluating a DFT single-point energy, and optimizing a DFT potential-energy surface. In an extended benzene/substituent study, specify a balanced observable and consistent geometry/energy protocols rather than subtracting unrelated total energies.

### 6.6.7. Save a reproducible record

Store the settings and geometries alongside the numerical tables. An XYZ file stores coordinates and element labels; the JSON record below also preserves charge, multiplicity, method, and preparation details. These files are local outputs, not external submissions.

In [ ]:
property_sensitivity.to_csv(OUTPUT_DIR / "dipole_sensitivity.csv")
water_comparison.to_csv(OUTPUT_DIR / "water_fixed_geometry.csv")
geometry_comparison.to_csv(OUTPUT_DIR / "water_geometry_comparison.csv")
frequency_table.to_csv(OUTPUT_DIR / "water_harmonic_frequencies.csv", index=False)
water_optimized.save_xyz_file(str(OUTPUT_DIR / "water_pbe_optimized.xyz"), True)
Chem.MolToXYZFile(butadiene_rdkit, str(OUTPUT_DIR / "butadiene_uff_input.xyz"))
record = {
    "versions": versions,
    "psi4_options": BASE_OPTIONS,
    "threads": 1,
    "memory": "512 MiB",
    "water": {"charge": 0, "multiplicity": 1,
              "initial_OH_angstrom": 1.0, "initial_HOH_degrees": 110.0,
              "fixed_geometry_functionals": ["PBE", "PBE0"], "optimization_functional": "PBE",
              "optimized_energy_hartree": float(optimized_energy),
              "maximum_gradient_hartree_per_bohr": maximum_gradient,
              "relaxation_energy_kJ_mol": float(relaxation_kj_mol),
              "frequency_derivative_level": "finite differences of analytic gradients"},
    "basis_check": {"functional": "PBE0", "basis": "def2-TZVP", "single_point": larger_basis_row},
    "grid_check": {"functional": "PBE0", "fine_radial_points": 150,
                   "fine_spherical_points": 974, "delta_energy_hartree": float(grid_delta_hartree)},
    "butadiene": {"smiles": "C=CC=C", "charge": 0, "multiplicity": 1,
                  "conformer_seed": 2026, "preparation": "ETKDGv3 then UFF",
                  "central_torsion_degrees": float(central_torsion_deg), "single_point": butadiene_row},
    "energy_convention": "Clamped-nuclei total energies including nuclear repulsion; no thermal corrections",
}
(OUTPUT_DIR / "calculation_record.json").write_text(json.dumps(record, indent=2) + "\n", encoding="utf-8")
psi4.core.clean()  # Remove Psi4-managed temporary scratch from this kernel's calculations.
print("Saved tables, geometries, calculation_record.json, and psi4.log under outputs/chapter06/.")

## 6.7. Exercises and self-checks

1. **Density normalization.** How many occupied spatial orbitals describe closed-shell water? What integral results if the factor of two is omitted from $n=2\sum_i|\phi_i|^2$?
2. **The theorem's scope.** Explain the difference between the universal $F[n]$ and the energy functional for a particular external potential. Why do the Hohenberg-Kohn theorems not give a ready-to-use exact functional?
3. **Exchange-correlation.** Identify the kinetic contribution included in $E_{xc}$. For one electron, explain why the exact functional must cancel the Hartree self-interaction.
4. **Functional families.** Classify PBE, r2SCAN, PBE0, a range-separated hybrid, and a double hybrid. Explain why range separation and dispersion corrections are not additional rungs of the ladder.
5. **Method comparison.** One fixed-geometry total energy is more negative than another. Why does that not show that its functional is more accurate? Name a useful reference observable and a way to validate its prediction.
6. **Independent numerical controls.** Use the dipole-sensitivity table. Is the grid change below the illustrative 0.001 D resolution? Does that establish 0.001 D accuracy? Which calculation probes basis incompleteness, and which changes the functional?
7. **Geometry and vibrations.** Explain why the optimized energy, small gradient, and positive vibrational modes answer three different questions. Why are tiny imaginary translational/rotational modes different from a substantial imaginary internal mode?
8. **Design a later research study (no additional calculation required).** Outline a study that optimizes several prepared butadiene conformers at one DFT level, verifies their stationary-point character, and compares their energies. State whether the reported difference includes zero-point/thermal corrections. Do not assume each starting conformer remains a distinct minimum.

9. **Reading the density figure.** Why should you not integrate the displayed plane and expect ten electrons?

<details><summary>Selected answers</summary>

1. Five doubly occupied spatial orbitals; omitting the factor gives five rather than ten electrons.
2. $F[n]$ contains kinetic and electron-electron interaction energies and is independent of the external potential; $E_v[n]$ also includes the external-potential contribution. Existence does not provide an explicit evaluable formula.
3. Kinetic correlation $T-T_s$ is included. One electron has no electron-electron pair repulsion, so its artificial Hartree self-energy must be canceled by $E_{xc}$.
4. GGA; meta-GGA; global hybrid; usually a hybrid on rung 4; unoccupied-orbital-dependent rung 5. These labels describe ingredients, not a universal accuracy ranking.
5. Approximate DFT energies are not generally variational upper bounds and use different functionals. Compare a defined property or balanced energy difference against appropriate experimental or high-level reference data.
6. Compare the grid-only change with 0.001 D. A small change supports numerical stability for this test; it does not bound functional, basis, geometry, or environment error. PBE0/def2-TZVP versus PBE0/def2-SVP probes basis sensitivity; PBE versus PBE0 changes the functional.
7. Energy lowering shows relaxation; the gradient checks stationarity; vibrational curvature distinguishes a local minimum from a saddle point within the approximation. Near-zero external modes contain numerical noise rather than an internal reaction coordinate.

8. Keep the method and numerical settings fixed, optimize and classify each candidate, remove duplicate minima, then compare consistent electronic or thermally corrected energy differences. The study design, not a larger job here, is the objective.
9. A two-dimensional density-slice integral does not give the total number of electrons: the slice omits the third spatial coordinate. Use the three-dimensional integral, or the equivalent basis-space trace already checked in the code.

</details>

**Takeaway:** specify the density functional approximation, basis, auxiliary basis, grid, charge/spin, geometry protocol, convergence criteria, and physical energy convention. A numerical result becomes useful when the model and the question are equally clear. Continue with [Chapter 7](Chapter07.ipynb).